In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
apikey = os.getenv("GOOGLE_API_KEY")
if apikey:
    print("yes")
else:
    print("no")



yes


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI as Gai
llm = Gai(
    model="gemini-2.5-flash-lite",
    temperature=0.5,
    max_output_tokens=300,
    top_k=3   
)

In [3]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a math expert.

Reason carefully before answering.
Keep reasoning concise and structured.
"""),
    
    ("human", """
Example:

Question:
What is 10 × 2?

Answer:
Step 1: Multiply 10 by 2
Step 2: 10 × 2 = 20
Final Answer: 20

---

Now solve the problem below.

Question:
{question}

Answer:
Step 1:
Step 2:
Step 3:
Final Answer:
""")
])

In [4]:
chain = prompt | llm

response = chain.invoke({
    "question": "What is 25 × 4?"
})

print(response.content)

Step 1: Multiply 25 by 4
Step 2: 25 × 4 = 100
Step 3: The result of the multiplication is 100.
Final Answer: 100


In [5]:
# pip install wikipedia arxiv langchain-community

from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain_community.tools.arxiv.tool import ArxivQueryRun
from langchain_community.utilities.arxiv import ArxivAPIWrapper

# Initialize the wrappers (No API keys required!)
wiki_wrapper = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=1000)
arxiv_wrapper = ArxivAPIWrapper(top_k_results=2, doc_content_chars_max=1000)

# Create the tools
wikipedia_tool = WikipediaQueryRun(api_wrapper=wiki_wrapper)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

# These tools can now be directly passed into an agent
tools = [wikipedia_tool, arxiv_tool]


 agent


In [7]:
query = input()
tool_output = arxiv_tool.run(query)
prompt = f"""
You are a research assistant. Answer the user's query based on the provided arXiv research papers.

User Query: 
"{query}"

Source Papers:
```text
{tool_output}"""

response = llm.invoke(prompt)
print(response.content)




The provided paper, "Optimal designs for comparing regression models with correlated observations" by Dette, Schorning, and Konstantinou (2016), focuses on the problem of efficiently comparing two regression curves when the observations are dependent. The authors propose a method to construct a pair of linear unbiased estimators and corresponding optimal designs for finite sample sizes. This approach aims to minimize the width of the confidence band for the difference between the estimated curves. The paper extends existing results to handle correlated observations and offers a practical and efficient solution, demonstrating its advantages through numerical examples.


In [ ]:
query = input()
tool_output = arxiv_tool.run(query)
prompt = f"""
You are a research assistant. Answer the user's query based on the provided arXiv research papers.

User Query: 
"{query}"

Source Papers:
```text
{tool_output}"""

for i in llm.stream(prompt):
    print(i.content, end="", flush =True)

